In [ ]:
# Run this first to clone the published DSA implementation.
from pathlib import Path
import subprocess
import sys
import os
import json
import hashlib
import shutil

REPO_URL = "https://github.com/sttruji/dynamic-state-anchoring.git"
REPO_REF = "main"  # Set a commit SHA here to reproduce a particular implementation.
REPO_DIR = Path("/content/dynamic-state-anchoring")

def git_output(*args):
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()

if not REPO_DIR.exists():
    try:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError("Check that the repository is reachable, then rerun. "
                           "Private repositories need authenticated Git access configured separately.") from exc
else:
    remote = git_output("remote", "get-url", "origin").rstrip("/").removesuffix(".git")
    if remote != REPO_URL.rstrip("/").removesuffix(".git"):
        raise RuntimeError("Existing checkout has a different remote; choose another REPO_DIR.")
    if git_output("status", "--porcelain"):
        raise RuntimeError("Existing checkout has local changes. Preserve them before updating it.")
subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
REPO_COMMIT = git_output("rev-parse", "HEAD")
if not (REPO_DIR / "requirements-colab.txt").is_file():
    raise RuntimeError("This ref does not contain the DSA implementation.")
print("DSA implementation:", REPO_COMMIT)


# Dynamic State Anchoring · FLT data-scale test

Choose **T4 GPU → Run all**. Defaults: **512 selected training records**, **1,024
updates per condition** (two passes), **K=8**, and all **500 original validation
questions**. This is a new experiment; previous 32-question pilots remain separate.
The model, corpus cache and dependency setup are reused when available.

All four conditions use the same pretrained GPT-2, FLT-inspired recurrence,
initial parameters, training records, losses, optimizer and update budget:
**static/no writes**, **writes every loop**, **periodic writes at loops 4 and 8**,
and **learned writes at threshold 0.5**. Each retains candidate and gate supervision.
The question remains accessible throughout. No gold trigger enters inference.

The earlier pilot demonstrated better gradient conditioning and training fitting
with FLT, but very low validation accuracy and no learned timing advantage. This
run tests generalization with more data and isolates the write-policy comparison.
It is one exploratory seed, not a benchmark claim or a full FLT reproduction.
See the [protocol](https://github.com/sttruji/dynamic-state-anchoring/blob/main/docs/FLT_DATA_SCALE.md).

The prior T4 timings suggest roughly **two hours for training updates** across
four arms, plus loading, validation, diagnostics and backups; this is an estimate.
Expensive diagnostics are capped at 64 fixed training records, eight intervention
records and two gradient probes. Change the budget in the configuration cell
before running subsequent cells; do not increase model size in this experiment.

Each arm runs locally and produces a full ZIP backup on Drive and a requested
browser download before the next arm. Keep these four backups and let transfers
finish. Verified finished arms are reused after reconnecting; partial arms restart
without a checkpoint-repair workflow. The final cell exports **one small review
ZIP to hand back**, including any recorded failures. Held-out test stays disabled.


In [ ]:
# Reuse Colab's CUDA/PyTorch stack; isolate project packages from the notebook kernel.
# The helper uses --without-pip --system-site-packages and targets this environment explicitly.
VENV_DIR = Path("/content/dsa-venv")
subprocess.run([sys.executable, str(REPO_DIR / "scripts/setup_colab.py"),
                "--venv-dir", str(VENV_DIR),
                "--requirements", str(REPO_DIR / "requirements-colab.txt")], check=True)
PYTHON = str(VENV_DIR / "bin/python")
print("Project Python:", PYTHON)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/dynamic-state-anchoring")
for name in ("corpus", "prepared", "runs", "reports", "cache/huggingface"):
    (PROJECT_DIR / name).mkdir(parents=True, exist_ok=True)
# Set before any subprocess imports Hugging Face packages.
os.environ["HF_HOME"] = str(PROJECT_DIR / "cache/huggingface")
os.environ["HF_HUB_CACHE"] = str(PROJECT_DIR / "cache/huggingface/hub")
os.environ["HUGGINGFACE_HUB_CACHE"] = os.environ["HF_HUB_CACHE"]
os.environ["TRANSFORMERS_CACHE"] = os.environ["HF_HUB_CACHE"]
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
print("Persistent storage:", PROJECT_DIR)


In [ ]:
# Fixed-budget data-scale test. Edit settings here before running subsequent cells.
RUN_MODE = "auto_pilot"  # Reuse verified finished arms; restart partial arms. "restart_all" forces all four.
EXPERIMENT = "gpt2-flt-data-v1"
EXPERIMENT_KIND = "data_scale"
BACKBONE = "yingfanbot/gsm-cot-gpt2"
MODEL_REVISION = "e0ef1887404adbf6c531d2cac7804694321f87bd"
DATA_REVISION = "e06a32ee5e4cd117171daeb4755d2a97ece62761"
SEED, TRAIN_LIMIT, LOOPS, WORKSPACE, STEPS = 7, 512, 8, 4, 1024
LEARNING_RATE, GRADIENT_ACCUMULATION = 3e-5, 1
CONDITIONS = {
    "static": {"recurrence_mode": "flt_memory", "write_mode": "static"},
    "every_loop": {"recurrence_mode": "flt_memory", "write_mode": "every_loop"},
    "periodic": {"recurrence_mode": "flt_memory", "write_mode": "periodic"},
    "learned": {"recurrence_mode": "flt_memory", "write_mode": "learned"},
}
ARMS = ["static", "every_loop", "periodic", "learned"]
SCREEN_LIMIT, SCREEN_DEPTHS = 1, [1, 8]
VALID_LIMIT = 0  # All 500 original validation questions; held-out test stays disabled.
MAX_NEW_TOKENS, COT_MAX_NEW_TOKENS = 64, 192
TRAIN_DIAGNOSTIC_LIMIT, GRADIENT_LIMIT, PROBE_LIMIT, CONTROL_LIMIT = 64, 2, 8, 8
DEPTH_GRID = [1, 4, 8]
DEVICE = "cuda"
DOWNLOAD_FULL_CHECKPOINTS = True
assert RUN_MODE in {"auto_pilot", "restart_all"}
assert 0 < TRAIN_DIAGNOSTIC_LIMIT <= min(TRAIN_LIMIT, 512)
assert 0 < GRADIENT_LIMIT <= TRAIN_DIAGNOSTIC_LIMIT and 0 < PROBE_LIMIT <= TRAIN_DIAGNOSTIC_LIMIT
assert STEPS >= TRAIN_LIMIT and ARMS and set(ARMS) == set(CONDITIONS)
training_settings = {key: globals()[key] for key in ("EXPERIMENT", "BACKBONE", "MODEL_REVISION", "DATA_REVISION",
    "SEED", "TRAIN_LIMIT", "LOOPS", "WORKSPACE", "STEPS", "LEARNING_RATE", "GRADIENT_ACCUMULATION", "ARMS", "CONDITIONS")}
experiment_hash = hashlib.sha256(json.dumps(training_settings, sort_keys=True).encode()).hexdigest()[:8]
RUN_NAME = f"{EXPERIMENT}-{REPO_COMMIT[:12]}-{experiment_hash}"
LEGACY_RUN_ROOT = PROJECT_DIR / "runs" / RUN_NAME
RUN_ROOT = Path("/content/dsa-pilot-working") / RUN_NAME
REPORT_ROOT = PROJECT_DIR / "reports" / RUN_NAME
SMOKE_REPORT = REPORT_ROOT / "unmodified-checkpoint-smoke.json"
CHECKPOINT_BACKUP_DIR = PROJECT_DIR / "bundles" / RUN_NAME / "checkpoints"
TEMPLATE_RUN_DIR = None
PREPARED_DRIVE = PROJECT_DIR / "prepared" / f"{REPO_COMMIT[:12]}-train{TRAIN_LIMIT}-k{LOOPS}-s{SEED}"
PREPARED_LOCAL = Path("/content/dsa-prepared") / PREPARED_DRIVE.name
EVAL_BUDGETS = [LOOPS]
review_settings = {key: globals()[key] for key in ("VALID_LIMIT", "MAX_NEW_TOKENS", "COT_MAX_NEW_TOKENS", "DEVICE",
    "TRAIN_DIAGNOSTIC_LIMIT", "GRADIENT_LIMIT", "PROBE_LIMIT", "CONTROL_LIMIT", "DEPTH_GRID", "SCREEN_LIMIT", "SCREEN_DEPTHS")}
review_hash = hashlib.sha256(json.dumps(review_settings, sort_keys=True).encode()).hexdigest()[:8]
BASE_REVIEW_NAME = REVIEW_NAME = f"data-review-v1-{REPO_COMMIT[:12]}-{review_hash}"
REVIEW_ROOT = REPORT_ROOT / REVIEW_NAME
REVIEW_ROOT.mkdir(parents=True, exist_ok=True)
import runpy, datetime
_run_logged = runpy.run_path(str(REPO_DIR / "scripts/colab_process.py"))["run_logged"]
def run_project(*args):
    return _run_logged([PYTHON, *map(str, args)], cwd=REPO_DIR,
                       env=os.environ.copy(), log_dir=REVIEW_ROOT / "logs")
status_path = REVIEW_ROOT / "session_status.json"
STAGE_STATUS = json.loads(status_path.read_text()) if status_path.exists() else {}
def record_stage(name, status, **details):
    STAGE_STATUS[name] = {"status": status, "utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), **details}
    temporary = status_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(STAGE_STATUS, indent=2) + "\n")
    temporary.replace(status_path)
def stage(name, *args):
    try:
        run_project(*args)
    except Exception as error:
        record_stage(name, "failed", error=str(error))
        print(f"STAGE FAILED: {name}. Error saved; continuing toward the review ZIP.", flush=True)
        return False
    record_stage(name, "passed")
    return True
config = {"RUN_NAME": RUN_NAME, "REVIEW_NAME": REVIEW_NAME, "REPO_COMMIT": REPO_COMMIT,
          "EXPERIMENT_KIND": EXPERIMENT_KIND, "RUN_MODE": RUN_MODE, **training_settings, **review_settings}
(REVIEW_ROOT / "protocol.json").write_text(json.dumps(config, indent=2) + "\n")
print(json.dumps(config, indent=2))
print(f"Budget: {len(ARMS)} arms × {STEPS} updates; {STEPS/TRAIN_LIMIT:.1f} passes through {TRAIN_LIMIT} selected training records.")
print("The previous T4 run suggests roughly two hours of training updates, plus evaluation/loading/backups; actual time may differ.")
print("Working checkpoints:", RUN_ROOT)
print("Completed full Drive backups:", CHECKPOINT_BACKUP_DIR)
print("Review outputs:", REVIEW_ROOT)
print("No partial checkpoint repair: complete compatible arms are reused; partial arms restart.")


The preflight checks CUDA, a tiny GPT-2 forward, the data-scale workflow and
FLT computation. The project environment keeps Colab's installed CUDA/PyTorch.
Failed stages are logged and included in the final review archive.


In [ ]:
runtime_code = r"""
import importlib.metadata, json, platform, sys
from pathlib import Path
import torch
from packaging.version import Version
from dsa.models.backbone import BackboneAdapter
assert (3, 10) <= sys.version_info[:2] < (3, 14), "Use a Python 3.10–3.13 runtime."
assert Version(torch.__version__.split('+')[0]) >= Version('2.7'), "Use PyTorch >=2.7."
requested_device, output = sys.argv[1:]
probe = {'requested_device': requested_device, 'python': platform.python_version(),
         'torch': torch.__version__, 'cuda_build': torch.version.cuda,
         'cuda_available': torch.cuda.is_available(), 'cuda_device_count': torch.cuda.device_count()}
print(json.dumps({'runtime_probe': probe}, indent=2), flush=True)
if requested_device == 'cuda' and not probe['cuda_available']:
    raise RuntimeError('CUDA is unavailable in this runtime. In Colab choose Runtime > Change runtime type > T4 GPU, '
                       'save/connect, then rerun setup. If a GPU is already allocated, inspect the runtime probe '
                       'and nvidia-smi output before changing packages.')
torch.set_num_threads(1)
model = BackboneAdapter.tiny(259).model.to(requested_device).eval()
with torch.no_grad():
    result = model(input_ids=torch.tensor([[3, 4, 5]], device=requested_device), use_cache=False)
assert bool(torch.isfinite(result.logits).all()), 'Nonfinite GPT-2 forward'
report = {'python': platform.python_version(), 'device': requested_device,
          'cuda_available': torch.cuda.is_available(), 'cuda': torch.version.cuda,
          'gpu': torch.cuda.get_device_name() if torch.cuda.is_available() else None,
          'gpu_memory_bytes': torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,
          'packages': {name: importlib.metadata.version(name) for name in
                       ('torch', 'transformers', 'tokenizers', 'huggingface_hub', 'numpy')},
          'tiny_gpt2_forward_passed': True}
Path(output).write_text(json.dumps(report, indent=2) + '\n')
print(json.dumps(report, indent=2))
"""
GPU_READY = stage("runtime-preflight", "-c", runtime_code, DEVICE, REVIEW_ROOT / "runtime.json")
CHECKS_READY = stage("review-correctness", "-m", "unittest", "discover", "-s", "tests", "-p", "test_data_scale_workflow.py", "-q")
print("Preflight complete; inspect recorded failures before interpreting results.")
MODEL_CHECKS_READY = stage("memory-injection-correctness", "-m", "unittest", "discover", "-s", "tests", "-p", "test_flt_memory.py", "-q")
GPU_READY = GPU_READY and CHECKS_READY and MODEL_CHECKS_READY


Corpus files are downloaded once to Drive and checked on reuse. The selected
512 training records are fixed by seed 7, with original source order restored;
unique question counts are reported separately because augmented records may
share a question. Validation membership is unchanged. Source quality filters
remain off, preserving the declared primary data policy.


In [ ]:
DATA_READY = stage("prepare-data", "scripts/prepare_dsa_pilot.py",
    "--storage-dir", PROJECT_DIR / "corpus", "--output-dir", PREPARED_DRIVE,
    "--train-limit", TRAIN_LIMIT, "--loops", LOOPS, "--seed", SEED,
    "--tokenizer-model", BACKBONE, "--revision", MODEL_REVISION)
if DATA_READY:
    try:
        shutil.copytree(PREPARED_DRIVE, PREPARED_LOCAL, dirs_exist_ok=True)
        prepared_manifest = json.loads((PREPARED_LOCAL / "manifest.json").read_text())
        record_stage("stage-cached-inputs", "passed", prepared_dir=str(PREPARED_DRIVE))
        counts = {name: entry["records"] for name, entry in prepared_manifest["splits"].items()}
        assert counts["train"] == TRAIN_LIMIT and counts["valid"] == 500
        selected = [json.loads(line) for line in (PREPARED_LOCAL / "train.tokenized.jsonl").read_text().splitlines() if line.strip()]
        print("Prepared records:", counts)
        print("Selected unique training questions:", len({r["question_hash"] for r in selected}))
        print("Source-quality exclusions remain off; no-event/overflow records retain main answer supervision.")
    except Exception as error:
        DATA_READY = False
        record_stage("stage-cached-inputs", "failed", error=str(error))
        print("Input staging failed:", error)


In [ ]:
SMOKE_READY = False
if GPU_READY and DATA_READY:
    SMOKE_READY = stage("original-checkpoint-smoke", "scripts/smoke_pretrained_dsa.py", "--backbone", BACKBONE,
        "--revision", MODEL_REVISION, "--device", DEVICE, "--output", SMOKE_REPORT)


A one-record initialization screen at K=1/8 checks every write policy with
the same parameters, without optimizer updates. Each arm then trains for exactly
1,024 updates at K=8. Optimizer, loss weights and clipping remain unchanged from
the successful FLT pilot. Periodic writes occur at loops 4 and 8.

Training uses VM disk. The completed checkpoint, tokenizer, optimizer/RNG state
and journal are archived together. A finished compatible arm can be restored
from its verified archive; partial runs restart. Training does not tune against
validation or choose a different checkpoint for each policy.


In [ ]:
SCREEN_READY = False
if GPU_READY and DATA_READY and SMOKE_READY:
    SCREEN_READY = stage("initialization-gradient-screen", "scripts/screen_recurrence.py",
        "--prepared-dir", PREPARED_LOCAL, "--output-dir", REVIEW_ROOT / "initialization-screen",
        "--device", DEVICE, "--smoke-report", SMOKE_REPORT,
        "--conditions", json.dumps({arm: CONDITIONS[arm] for arm in ARMS}),
        "--limit", SCREEN_LIMIT, "--depths", *SCREEN_DEPTHS, "--seed", SEED, "--workspace", WORKSPACE)
    if SCREEN_READY:
        screen_report = json.loads((REVIEW_ROOT / "initialization-screen/report.json").read_text())
        SCREEN_READY = screen_report["finite"] and screen_report["model_state_unchanged"]
        print("Initialization answer-gradient norms:", screen_report["median_answer_gradient_norms"])


In [ ]:
training_reports, arm_checkpoints, READY_ARMS = {}, {}, []
initial_sha = globals().get("screen_report", {}).get("initial_state_sha256")
BACKUP_DOWNLOADS_STARTED = set()

def download_full_checkpoints():
    from google.colab import files
    paths = {arm: Path(result["local_full_backup"]) for arm, result in arm_checkpoints.items()
             if Path(result["local_full_backup"]).is_file()}
    for arm, path in paths.items():
        if arm in BACKUP_DOWNLOADS_STARTED: continue
        print(f"FULL BACKUP {arm}: {path.stat().st_size / 1e9:.2f} GB. Keep this ZIP and wait for the download to finish.")
        files.download(str(path))
        BACKUP_DOWNLOADS_STARTED.add(arm)
        record_stage(f"backup-download-{arm}", "passed", request_started_only=True)
    return bool(paths) and set(paths) <= BACKUP_DOWNLOADS_STARTED

for arm in ARMS:
    run_dir = RUN_ROOT / arm
    if GPU_READY and DATA_READY and SMOKE_READY and SCREEN_READY:
        out = REVIEW_ROOT / arm / "checkpoint-status.json"
        args = ["scripts/ensure_pilot_arm.py", "--prepared-dir", PREPARED_LOCAL,
            "--working-dir", run_dir, "--legacy-dir", LEGACY_RUN_ROOT / arm,
            "--backup-dir", CHECKPOINT_BACKUP_DIR, "--report-path", out, "--arm", arm,
            "--steps", STEPS, "--device", DEVICE, "--smoke-report", SMOKE_REPORT,
            "--seed", SEED, "--workspace", WORKSPACE, "--learning-rate", LEARNING_RATE,
            "--gradient-accumulation", GRADIENT_ACCUMULATION,
            "--write-mode", CONDITIONS[arm]["write_mode"], "--recurrence-mode", CONDITIONS[arm]["recurrence_mode"]]
        if TEMPLATE_RUN_DIR: args += ["--template-dir", TEMPLATE_RUN_DIR]
        if initial_sha: args += ["--expected-initial-sha", initial_sha]
        if RUN_MODE == "restart_all": args += ["--force-restart"]
        if stage(f"ensure-{arm}", *args):
            result = json.loads(out.read_text())
            arm_checkpoints[arm] = result
            initial_sha = result["initial_state_sha256"]
            READY_ARMS.append(arm)
            if DOWNLOAD_FULL_CHECKPOINTS:
                try: download_full_checkpoints()
                except Exception as error:
                    record_stage(f"backup-download-{arm}", "failed", error=str(error))
                    print("Full backup remains on VM/Drive; final cell retries:", error)
            record_stage(f"drive-backup-{arm}", "failed" if result["backup_error"] else "passed",
                         error=result["backup_error"], mounted_readback_only=True)
    if arm in READY_ARMS:
        report_path = run_dir / "train_report.json"
        training_reports[arm] = json.loads(report_path.read_text()) if report_path.exists() else {
            "step": json.loads((run_dir / "latest.json").read_text())["step"]}
        print(f"{arm}: ready at step {training_reports[arm]['step']}")
# Bind review outputs to the actual checkpoint bytes. A restart gets a fresh review
# location whenever its state differs; stale predictions never masquerade as new.
if arm_checkpoints:
    checkpoint_identity = {arm: {name: entry["sha256"] for name, entry in result["backup_files"].items()
        if name.startswith("checkpoint-")} for arm, result in arm_checkpoints.items()}
    checkpoint_hash = hashlib.sha256(json.dumps(checkpoint_identity, sort_keys=True).encode()).hexdigest()[:12]
    REVIEW_NAME = BASE_REVIEW_NAME + "-" + checkpoint_hash
    selected_review = REPORT_ROOT / REVIEW_NAME
    selected_review.mkdir(parents=True, exist_ok=True)
    for path in REVIEW_ROOT.rglob("*"):
        relative = path.relative_to(REVIEW_ROOT)
        if selected_review != REVIEW_ROOT and path.is_file() and (relative.parts[0] == "logs" or path.name in {
                "checkpoint-status.json", "runtime.json", "session_status.json", "protocol.json"} or relative.parts[0] == "initialization-screen"):
            target = selected_review / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)
    REVIEW_ROOT = selected_review
    status_path = REVIEW_ROOT / "session_status.json"
    config["REVIEW_NAME"] = REVIEW_NAME
    (REVIEW_ROOT / "protocol.json").write_text(json.dumps(config, indent=2) + "\n")
    print("Checkpoint-bound review:", REVIEW_ROOT)

FULL_BACKUP_DOWNLOAD_STARTED = bool(arm_checkpoints) and set(arm_checkpoints) <= BACKUP_DOWNLOADS_STARTED


## Fixed validation protocol

Every arm evaluates all **500 original validation questions** at K=8 using the
same greedy decoder and 64-token cap. Saved predictions can resume, and review
paths are bound to the actual completed checkpoint bytes. The original numeric
scorer is retained; raw text is exported so incidental numeric matches can be
reviewed. Test questions remain reserved.


In [ ]:
def checkpoint_step(arm):
    return json.loads((RUN_ROOT / arm / "latest.json").read_text())["step"]
def evaluation_dir(arm, split, budget, limit):
    return REVIEW_ROOT / arm / f"{split if split == 'test' else 'validation'}-step{checkpoint_step(arm)}-k{budget}-n{limit}-g{MAX_NEW_TOKENS}"
validation_reports = []
if GPU_READY and DATA_READY:
    for arm in READY_ARMS:
        for budget in EVAL_BUDGETS:
            out = evaluation_dir(arm, "valid", budget, VALID_LIMIT)
            if stage(f"validation-{arm}-k{budget}", "scripts/evaluate_dsa.py", "--prepared-dir", PREPARED_LOCAL,
                     "--run-dir", RUN_ROOT / arm, "--output-dir", out, "--split", "valid",
                     "--loop-budget", budget, "--limit", VALID_LIMIT, "--max-new-tokens", MAX_NEW_TOKENS, "--device", DEVICE):
                validation_reports.append({"arm": arm, **json.loads((out / "report.json").read_text())})

## Bounded training diagnostics

Generation and deterministic teacher-forced forward profiles cover **64 fixed
training records**, selected without replacement with the training seed and
restored to source order. Their IDs and the full training denominator are saved.
This is a sampled training diagnostic, not accuracy on all 512 records.

The first eight selected diagnostic rows receive paired depth K=1/4/8, no-write,
every-write and zero-anchor/workspace/memory interventions. The first two receive
answer-gradient depth probes and weighted joint-loss gradients. The same forward
graph measures competing losses. Diagnostics use no optimizer updates and verify
unchanged model weights. Their runtime does not grow with the training dataset.

Continuous latent states are not decoded as thoughts. Intervention sensitivity,
alignment and gate event discrimination are evidence about model behavior, not
proof of semantic reasoning. Generalization is assessed on validation.


In [ ]:
diagnostic_reports = {}
if GPU_READY and DATA_READY:
    for arm in READY_ARMS:
        out = REVIEW_ROOT / arm / "diagnostics"
        if stage(f"diagnostics-{arm}", "scripts/diagnose_dsa_pilot.py", "--prepared-dir", PREPARED_LOCAL,
                 "--run-dir", RUN_ROOT / arm, "--output-dir", out, "--device", DEVICE,
                 "--training-limit", TRAIN_DIAGNOSTIC_LIMIT, "--gradient-limit", GRADIENT_LIMIT, "--probe-limit", PROBE_LIMIT,
                 "--depth-grid", *DEPTH_GRID, "--max-new-tokens", MAX_NEW_TOKENS, "--comprehensive"):
            diagnostic_reports[arm] = json.loads((out / "report.json").read_text())


## Small explicit-CoT control

The original cached model and each trained backbone answer the same eight train
and eight validation questions through the ordinary explicit language route.
The 192-token cap differs from latent decoding, so this checks route capability
and possible degradation without claiming matched inference compute. No new
teacher model is trained or downloaded for this control.


In [ ]:
control_reports = {}
if GPU_READY and DATA_READY:
    conditions = [("original", None)] + [(arm, RUN_ROOT / arm) for arm in READY_ARMS]
    for name, run_dir in conditions:
        out = REVIEW_ROOT / "original-explicit-control" if run_dir is None else REVIEW_ROOT / name / "explicit-control"
        args = ["scripts/pilot_explicit_controls.py", "--prepared-dir", PREPARED_LOCAL,
                "--output-dir", out, "--device", DEVICE, "--limit", CONTROL_LIMIT,
                "--max-new-tokens", COT_MAX_NEW_TOKENS]
        if run_dir is not None: args += ["--run-dir", run_dir]
        if stage(f"explicit-control-{name}", *args): control_reports[name] = json.loads((out / "report.json").read_text())

## Final held-out evaluation — leave disabled during development

Freeze the training setup, checkpoint choice, decoding settings and loop budgets
using validation before enabling this cell. It evaluates **every one of the 1,319
test questions** for every declared arm/budget with the pinned LOTUS scorer. Test
answers do not select checkpoints or tune any setting. A small-subset training
run still remains a pilot even if it evaluates the full test split.


In [ ]:
RUN_TEST = False  # Enable only after freezing the complete evaluation protocol.
final_reports = []
if RUN_TEST:
    for arm in ARMS:
        for budget in EVAL_BUDGETS:
            out = evaluation_dir(arm, "test", budget, 0)
            run_project("scripts/evaluate_dsa.py", "--prepared-dir", PREPARED_LOCAL,
                        "--run-dir", RUN_ROOT / arm, "--output-dir", out,
                        "--split", "test", "--loop-budget", budget, "--limit", 0,
                        "--max-new-tokens", MAX_NEW_TOKENS, "--device", DEVICE, "--final-test")
            result = json.loads((out / "report.json").read_text())
            assert result["total"] == 1319 and result["all_source_examples_retained"]
            final_reports.append({"arm": arm, **result})
else:
    print("Held-out test evaluation is disabled. Use validation for pilot decisions.")


In [ ]:
print(f"{'arm':12} {'split':6} {'K':>3} {'n':>5} {'accuracy':>9} {'invalid':>9} {'sec/q':>9}")
for report in globals().get("validation_reports", []) + globals().get("final_reports", []):
    print(f"{report['arm']:12} {report['split']:6} {report['loop_budget']:3d} {report['total']:5d} "
          f"{report['accuracy']:9.3%} {report['invalid_answer_rate']:9.3%} {report['average_latency_seconds']:9.3f}")
training_profiles = {}
for arm, report in training_reports.items():
    journal = RUN_ROOT / arm / "metrics.jsonl"
    metrics = []
    if journal.exists():
        with journal.open() as handle:
            for line in handle:
                if not line.strip(): continue
                try: metrics.append(json.loads(line))
                except json.JSONDecodeError:
                    print("Incomplete/corrupt metric line:", journal)
    measured_seconds = sum(row["seconds"] for row in metrics)
    profile = {"completed_steps": report["step"], "measured_update_seconds": measured_seconds,
               "updates_per_second": len(metrics) / measured_seconds if measured_seconds else None,
               "peak_process_rss_bytes": max((row["peak_process_rss_bytes"] for row in metrics), default=0),
               "peak_cuda_allocated_bytes": max((row["peak_cuda_allocated_bytes"] or 0 for row in metrics), default=0),
               "first_total_loss": metrics[0]["total"] if metrics else None,
               "last_total_loss": metrics[-1]["total"] if metrics else None,
               "mean_write_rate": sum(row["write_rate"] for row in metrics) / len(metrics) if metrics else None}
    training_profiles[arm] = profile
    print(arm, "training profile:", json.dumps(profile, indent=2))
for arm, result in globals().get("diagnostic_reports", {}).items():
    print(arm, "training generation:", result["prediction_reports"]["normal_training_set"])
for name, result in globals().get("control_reports", {}).items():
    print(name, "explicit control:", result["splits"])
prepared_path = PREPARED_LOCAL / "manifest.json"
summary = {"config": config, "training_profiles": training_profiles,
           "prepared_manifest_sha256": hashlib.sha256(prepared_path.read_bytes()).hexdigest() if prepared_path.exists() else None,
           "training": training_reports, "validation": globals().get("validation_reports", []),
           "final_test": globals().get("final_reports", []), "stage_status": globals().get("STAGE_STATUS", {}),
           "purpose": "fixed-budget data-scale experiment; one seed; no automatic research claim"}
summary_path = globals().get("REVIEW_ROOT", REPORT_ROOT) / "notebook-summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n")
print("Persistent summary:", summary_path)
print("Continue to the ZIP export cell, including after recorded GPU-stage failures.")


## Final review ZIP and completed checkpoint backups

Hand back the **small review ZIP**. It contains all four policies, all 500
validation outputs per arm, actual training journals, sampled training outputs,
controller targets/probabilities/AUC, paired validation counts, gradient and
memory interventions, explicit controls, configuration, source and artifact
hashes, runtime reports and failures. Sampled training metrics are labeled with
their denominator; the old 80% memorization heuristic does not block this run.

Keep the **four full checkpoint ZIPs** separately; they include weights and
training state. Mounted Drive readback is checked, but completed browser downloads
and cloud durability cannot be established by this notebook. Wait for transfers
before disconnecting. No larger model run starts automatically.


In [ ]:
from google.colab import files
BUNDLE_PATH = PROJECT_DIR / "bundles" / f"{RUN_NAME}-{REVIEW_NAME}.zip"
if stage("review-export", "scripts/package_pilot_review.py", "--project-dir", PROJECT_DIR,
            "--run-name", RUN_NAME, "--review-name", REVIEW_NAME, "--run-root", RUN_ROOT,
            "--prepared-dir", PREPARED_DRIVE, "--output", BUNDLE_PATH, "--arms", *ARMS):
    print("SMALL REVIEW ZIP:", BUNDLE_PATH)
    from zipfile import ZipFile
    with ZipFile(BUNDLE_PATH) as archive:
        manifest = json.loads(archive.read("bundle_manifest.json"))
    print("Evidence complete:", manifest["evidence_complete"])
    print("Hand this ZIP back for analysis. Keep the four separate full checkpoint ZIPs as your backups.")
    files.download(str(BUNDLE_PATH))
if DOWNLOAD_FULL_CHECKPOINTS and not FULL_BACKUP_DOWNLOAD_STARTED:
    try: FULL_BACKUP_DOWNLOAD_STARTED = download_full_checkpoints()
    except Exception as error: print("Full archives remain on the VM and Drive; download error:", error)
print("Recorded failed stages:", {name: entry for name, entry in STAGE_STATUS.items() if entry['status'] != 'passed'})
